# Coin Detection & Valuation using Computer Vision

This notebook implements a computer vision pipeline to detect and count coins in images, then calculate the total monetary value. Three coin types are detected:
- **golden** — worth 1 unit
- **red** — worth 2 units
- **star** — worth 5 units

The approach uses HSV color masking, morphological operations, and contour filtering.

## 1. Imports

In [ ]:
import numpy as np
import cv2
import csv
import os
import matplotlib.pyplot as plt

## 2. Configuration

### 2.1 Reference RGB Colors

Each coin type has a representative RGB color used as the center of its HSV detection range.

In [ ]:
golden_rgb = [164, 143, 29]  # Golden coin
red_rgb = [163, 67, 41]   # Red coin
star_rgb = [177, 167, 48]  # Star coin

### 2.2 Area Ranges, Kernels & Values

Valid contour area ranges prevent noise (too small) and merged blobs (too large) from being counted as coins.

In [ ]:
coins_area_range = {
    'golden': (150, 2500),
    'star': (4500, 8500),
    'red': (100, 2200)
}

# Elliptical structuring elements sized proportionally to each coin type
coin_kernels = {
    'golden': cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
    'star': cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17)),
    'red': cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
}

coin_values = {
    'golden': 1,
    'red': 2,
    'star': 5
}

## 3. HSV Range Generation

HSV (Hue, Saturation, Value) color space is more robust to lighting changes than RGB. Each coin type's detection range is built by converting its reference RGB color to HSV and expanding it by tolerance values.

In [ ]:
def make_hsv_range(rgb, h_tol, s_tol, v_tol):
    """
    Convert an RGB color to HSV and build a detection range
    by expanding ± tolerance in each channel.

    Parameters
    ----------
    rgb   : list[int]  Reference color as [R, G, B]
    h_tol : int        Hue tolerance (0–179 in OpenCV)
    s_tol : int        Saturation tolerance (0–255)
    v_tol : int        Value tolerance (0–255)

    Returns
    -------
    lower, upper : np.ndarray  HSV lower and upper bounds
    """
    hsv = cv2.cvtColor(np.uint8([[rgb]]), cv2.COLOR_RGB2HSV)[0][0]
    hsv = hsv.astype(int)

    lower = np.array([
        max(hsv[0] - h_tol, 0),
        max(hsv[1] - s_tol, 0),
        max(hsv[2] - v_tol, 0)
    ], dtype=np.uint8)

    upper = np.array([
        min(hsv[0] + h_tol, 179),
        min(hsv[1] + s_tol, 255),
        min(hsv[2] + v_tol, 255)
    ], dtype=np.uint8)

    return lower, upper


coin_hsv_ranges = {
    'golden': make_hsv_range(golden_rgb, h_tol=8,  s_tol=55, v_tol=55),
    'red': make_hsv_range(red_rgb, h_tol=5,  s_tol=35, v_tol=40),
    'star': make_hsv_range(star_rgb, h_tol=7,  s_tol=64, v_tol=56)
}

# Print computed HSV ranges for inspection
for coin, (lo, hi) in coin_hsv_ranges.items():
    print(f"{coin:8s}  lower={lo}  upper={hi}")

## 4. Image Processing Pipeline

### 4.1 Loading & Preprocessing

In [ ]:
def load_image(path):
    """Load an image from disk and convert BGR → RGB."""
    return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)


def preprocess_image(img):
    """
    Apply a mild Gaussian blur to reduce high-frequency noise
    before color segmentation.
    """
    return cv2.GaussianBlur(img, (3, 3), 0)

### 4.2 Coin Detection

The detection pipeline for each coin type:

1. **Gaussian blur** — suppress noise
2. **HSV masking** — isolate pixels matching the coin color
3. **Morphological opening** — remove small specks
4. **Morphological closing** — fill gaps within coin blobs
5. **Distance transform** *(golden/red only)* — separate touching coins
6. **Contour detection** — find candidate blobs
7. **Area + aspect ratio filtering** — reject non-coin shapes

In [ ]:
def count_coins(img, coin_type='golden', debug=False):
    """
    Count coins of a given type in an RGB image.

    Parameters
    ----------
    img       : np.ndarray  Input image in RGB format
    coin_type : str         One of 'golden', 'red', 'star'
    debug     : bool        If True, display intermediate results

    Returns
    -------
    int  Number of detected coins
    """
    img_processed = preprocess_image(img)
    hsv = cv2.cvtColor(img_processed, cv2.COLOR_RGB2HSV)

    # --- Color masking ---
    mask = cv2.inRange(hsv, *coin_hsv_ranges[coin_type])

    # --- Morphological cleanup ---
    kernel = coin_kernels[coin_type]
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # --- Distance transform to separate touching small coins ---
    if coin_type in ('golden', 'red'):
        dist_transform = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
        _, sure_fg = cv2.threshold(
            dist_transform, 0.2 * dist_transform.max(), 255, 0
        )
        mask = np.uint8(sure_fg)

    # --- Contour detection ---
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    # --- Filter by area and aspect ratio ---
    valid_contours = []
    min_area, max_area = coins_area_range[coin_type]
    for contour in contours:
        area = cv2.contourArea(contour)
        if not (min_area <= area <= max_area):
            continue
        _, _, w, h = cv2.boundingRect(contour)
        aspect_ratio = float(w) / h if h > 0 else 0
        if 0.5 < aspect_ratio < 2.0:
            valid_contours.append(contour)

    # --- Optional debug visualisation (LLM-assisted) ---
    if debug:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        debug_img = img.copy()
        cv2.drawContours(debug_img, valid_contours, -1, (0, 255, 0), 2)
        axes[0].imshow(debug_img)
        axes[0].set_title(f'Detected {coin_type} coins ({len(valid_contours)})')
        axes[0].axis('off')
        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title(f'Mask — {coin_type}')
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

    return len(valid_contours)

## 5. Evaluation

### 5.1 Load Ground Truth

The ground truth CSV (`coin_value_count.csv`) contains columns `image_name` and `total_value`.

In [ ]:
DIR = "data"

ground_truth = {}
ground_truth_file = os.path.join(DIR, "coin_value_count.csv")

with open(ground_truth_file, "r") as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        image_name = row[0]
        true_value = float(row[1])
        ground_truth[image_name] = true_value

print(f"Loaded ground truth for {len(ground_truth)} images.")

### 5.2 Run Predictions

In [ ]:
predicted_scores = {}
coin_breakdown = {}  # store per-coin counts for analysis

for image_name in ground_truth.keys():
    img_path = os.path.join(DIR, image_name)
    img = load_image(img_path)

    total_value = 0
    counts = {}
    for coin_type, value in coin_values.items():
        n = count_coins(img, coin_type=coin_type, debug=False)
        counts[coin_type] = n
        total_value += n * value

    predicted_scores[image_name] = total_value
    coin_breakdown[image_name] = counts

print("Predictions complete.")

### 5.3 Compute Mean Absolute Error (MAE)

In [ ]:
errors = [
    abs(ground_truth[name] - predicted_scores.get(name, 0))
    for name in ground_truth
]

mae = sum(errors) / len(errors)
print(f"Mean Absolute Error (MAE): {mae:.2f}")

### 5.4 Results Table

In [ ]:
print(f"{'Image':<30} {'True':>6} {'Pred':>6} {'Error':>6}")
print("-" * 52)
for name in sorted(ground_truth.keys()):
    true = ground_truth[name]
    pred = predicted_scores.get(name, 0)
    err  = abs(true - pred)
    print(f"{name:<30} {true:>6.1f} {pred:>6.1f} {err:>6.1f}")
print("-" * 52)
print(f"{'MAE':<30} {'':>6} {'':>6} {mae:>6.2f}")

## 6. Visualisation

### 6.1 Prediction vs Ground Truth

In [ ]:
names  = sorted(ground_truth.keys())
trues  = [ground_truth[n] for n in names]
preds  = [predicted_scores.get(n, 0) for n in names]

x = range(len(names))
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([i - 0.2 for i in x], trues, width=0.4, label='Ground Truth', color='steelblue')
ax.bar([i + 0.2 for i in x], preds, width=0.4, label='Predicted',    color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Total Coin Value')
ax.set_title(f'Ground Truth vs Predicted  (MAE = {mae:.2f})')
ax.legend()
plt.tight_layout()
plt.show()

### 6.2 Debug — Visualise a Single Image

In [ ]:
# Change this to any image name from the dataset
SAMPLE_IMAGE = list(ground_truth.keys())[0]

img = load_image(os.path.join(DIR, SAMPLE_IMAGE))
print(f"Debugging: {SAMPLE_IMAGE}")

for coin_type in coin_values:
    count_coins(img, coin_type=coin_type, debug=True)